In [ ]:
import re
import glob
import numpy as np
import pandas as pd
from scipy import stats, signal
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import librosa
import joblib
import os
import pywt

DATA_DIR = "dataset/wrench"
OUT_CSV = "features_dataset.csv"

In [ ]:
def parse_filename(fname):
    # format: C6--X--YYYY.csv
    base = Path(fname).name

    # no test object case
    m_no = re.match(r"C6--no--(\d{5})\.csv", base)
    if m_no:
        iteration = int(m_no.group(1))
        return {"presence": 0, "object": None, "row": np.nan, "col": np.nan, "iteration": iteration}

    # with test object case
    m_obj = re.match(r"C6--(.+?)--(\d+)--(\d+)--(\d{5})\.csv", base)
    if m_obj:
        obj, row, col, iteration = m_obj.groups()
        return {
            "presence": 1,
            "object": obj,
            "row": int(row),
            "col": int(col),
            "iteration": int(iteration)
        }

    # If nothing matched
    return {"presence": None, "object": None, "row": None, "col": None, "iteration": None}

# Calculating Features
## Statistical Measurement
**mean** — average value over a specified time; <br>
**standard deviation** — statistical measure of its AC component, indicating how much the signal fluctuates or deviates from its average (DC) value; <br>
**variance** — a statistical measure of the spread or variability of the current's values around its mean; <br>
**RMS** — effective value, constant direct current that would dissipate the same power in a resistive load; <br>
**Peak-to-Peak** — the difference between its maximum positive peak and its maximum negative peak during one cycle; <br>
**Median** — point above and below which 50% of the observed data falls; <br>
**Interquartile Range** — measure of spread that tells you the range of the middle 50% of data, calculated by subtracting the first quartile (Q1) from the third quartile (Q3); <br>
**Skewness** — describes the asymmetry of its waveform's probability distribution; <br>
**Kurtosis** — the sharpness of the peak of a frequency-distribution curve; <br>
**Crest Factor** — the ratio of a signal's peak value to its Root Mean Square (RMS) value; <br>
**Impulse Factor** — measures the "peakiness" of a signal by comparing its peak amplitude to the mean of its absolute values; <br>
**Shape Factor** — the ratio of the Root Mean Square (RMS) to the mean of the absolute values of a signal, indicating how the signal's energy is distributed; <br>
**Clearance Factor** — the ratio of a signal's peak value to the squared mean of the square roots of its absolute amplitudes; <br>

## Spectral features
Sample rate — 2.5 GS/s; <br>

**Estimate power spectral density using Welch’s method** <br>
Welch’s method computes an estimate of the power spectral density by dividing the data into overlapping segments, computing a modified periodogram for each segment and averaging the periodograms. <br>
Segment length for Welch method (nperseg) -> largest power of 2 <= N <br>

**Spectral Centroid** — indicates where the center of mass of the spectrum is located; <br>
**Spectral bandwidth** — Measures how spread out the spectrum is around its centroid; <br>
**Spectral entropy** — disorder or complexity of the spectrum; <br>
**Spectral Flatness** — Ratio of geometric mean to arithmetic mean of the spectrum; Indicates how “flat” the spectrum is; <br>
**Spectral Rolloff** — Finds the frequency below which a certain percentage(85%) of total power is contained. <br>
**Total Power** — Computes the area under the PSD curve; <br>

In [ ]:
def compute_features_from_array(t, x, rolloff_pct=0.85):
    # t, x are numpy arrays
    N = len(x)
    feats = {}
    
    # basic stats
    feats['time_mean'] = np.mean(x)
    feats['time_std'] = np.std(x) # standard deviation
    feats['var'] = np.var(x)
    rms = np.sqrt(np.mean(x**2))
    feats['rms'] = rms
    feats['ptp'] = np.ptp(x)
    feats['iqr'] = np.percentile(x,75) - np.percentile(x,25) # interquartile range
    feats['time_skew'] = stats.skew(x)
    feats['time_kurtosis'] = stats.kurtosis(x, fisher=False)
    feats['crest_factor'] = np.max(np.abs(x)) / feats['rms'] if feats['rms'] != 0 else 0
    mean_abs = np.mean(np.abs(x))
    feats['impulse_factor'] = np.max(np.abs(x)) / mean_abs if mean_abs != 0 else np.nan
    feats['shape_factor'] = rms / mean_abs if mean_abs != 0 else np.nan
    mean_sqrt = np.mean(np.sqrt(np.abs(x)))
    feats['clearance_factor'] = np.max(np.abs(x)) / (mean_sqrt**2) if mean_sqrt != 0 else np.nan
    
    # spectral features
    fs = 1.0 / np.mean(np.diff(t))
    nperseg = 2 ** int(np.floor(np.log2(N)))

    f, Pxx = signal.welch(x, fs=fs, nperseg=nperseg) # estimates the power spectral density (PSD) — a smoother version of FFT.
    Pxx = np.maximum(Pxx, 1e-20) # power at each frequency.

    # 1. Dominant Frequency (Fundamental)
    idx_max = np.argmax(Pxx)
    f0 = f[idx_max] # The fundamental frequency
    p0 = Pxx[idx_max] # Power of fundamental
    feats['peak_freq'] = f0
    feats['peak_power'] = p0

    # 2. Harmonic Features (THD and Specific Harmonics)
    # Look for harmonics at 2*f0, 3*f0, etc. Use a small window around the target freq to account for spectral leakage
    harmonics_power = []
    num_harmonics = 5
    
    def get_power_at_freq(target_f, f_arr, p_arr, window_points=2):
        # Find index closest to target frequency
        idx = (np.abs(f_arr - target_f)).argmin()
        # Sum power in a small window around that index
        start = max(0, idx - window_points)
        end = min(len(p_arr), idx + window_points + 1)
        return np.sum(p_arr[start:end])

    fund_energy = get_power_at_freq(f0, f, Pxx)

    total_harmonic_energy = 0
    for h in range(2, num_harmonics + 2):
        h_freq = h * f0
        if h_freq < f[-1]: # Ensure we are within Nyquist
            h_pwr = get_power_at_freq(h_freq, f, Pxx)
            harmonics_power.append(h_pwr)
            total_harmonic_energy += h_pwr
            
            # Feature: Individual Harmonic Ratios (e.g., 2nd harmonic vs Fundamental)
            feats[f'harmonic_{h}_ratio'] = h_pwr / (fund_energy + 1e-20)
        else:
            harmonics_power.append(0)
    
    # THD Calculation
    feats['thd'] = np.sqrt(total_harmonic_energy) / (np.sqrt(fund_energy) + 1e-20)

    # 3. Signal to Noise Ratio (Approximate)
    # Total power minus fundamental and harmonics
    total_spec_energy = np.sum(Pxx)
    noise_energy = total_spec_energy - fund_energy - total_harmonic_energy
    noise_energy = np.maximum(noise_energy, 1e-20)
    feats['snr'] = 10 * np.log10(fund_energy / noise_energy)
    
    # 4. Spectral shape
    spec_sum = float(np.sum(Pxx))
    feats['spec_centroid'] = float(np.sum(f * Pxx) / (spec_sum + 1e-20))
    centroid = feats['spec_centroid']
    feats['spec_bandwidth'] = float(np.sqrt(np.sum(((f - centroid) ** 2) * Pxx) / (spec_sum + 1e-20)))

    spec_std = np.sqrt(np.sum(((f - centroid) ** 2) * Pxx) / spec_sum)
    feats['spec_skew'] = float(np.sum(((f - centroid) ** 3) * Pxx) / (spec_sum * (spec_std ** 3) + 1e-20))
    
    # Spectral Kurtosis
    feats['spec_kurtosis'] = float(np.sum(((f - centroid) ** 4) * Pxx) / (spec_sum * (spec_std ** 4) + 1e-20))

    # spectral entropy and flatness
    Pnorm = Pxx / (np.sum(Pxx) + 1e-20)
    Pnorm = np.maximum(Pnorm, 1e-20)
    feats['spec_entropy'] = float(-np.sum(Pnorm * np.log2(Pnorm)))
    feats['spec_flatness'] = float(np.exp(np.mean(np.log(np.maximum(Pxx, 1e-20)))) / (np.mean(Pxx) + 1e-20))

    # spectral rolloff (freq below which rolloff_pct of power resides)
    cumsum = np.cumsum(Pxx)
    target = rolloff_pct * cumsum[-1]
    roll_idx = np.searchsorted(cumsum, target)
    feats['spec_rolloff'] = float(f[roll_idx] if roll_idx < len(f) else f[-1])

    feats['total_power'] = float(np.trapezoid(Pxx, f))

    # --- WAVELET FEATURES ---
    # We use 'db4' because it roughly matches the shape of transient spikes.
    # Level=5 allows us to break down the signal into 5 levels of detail.
    wavelet_name = 'db4'
    decomp_level = 5

    # Perform Discrete Wavelet Transform
    # coeffs will be a list: [cA5, cD5, cD4, cD3, cD2, cD1]
    # cA = Approximation (Low Freq), cD = Detail (High Freq)
    coeffs = pywt.wavedec(x, wavelet_name, level=decomp_level)
    
    # Calculate Total Energy of the signal (sum of squared coefficients)
    total_wav_energy = sum(np.sum(np.array(c)**2) for c in coeffs)

    # Extract features for each decomposition level
    # Level 0 is the Approximation (Deepest Low Freq - The main carrier)
    # Levels 1-5 are Details (High Freq - The noise/glitches)
    
    # Reverse the list so index 1 matches Detail Level 1 (High Freq)
    # Current coeffs: [cA_n, cD_n, ..., cD_1]
    # Let's organize explicitly:
    feat_names = ['approx'] + [f'detail_{i}' for i in range(decomp_level, 0, -1)]
    
    for i, c in enumerate(coeffs):
        name = feat_names[i]
        energy = np.sum(np.array(c)**2)
        
        # 1. Relative Energy
        # (How much of the signal power is in this specific frequency band?)
        feats[f'wav_energy_{name}'] = energy / (total_wav_energy + 1e-20)
        
        # 2. Wavelet Entropy
        # (Is the energy concentrated in a few spikes or spread out?)
        # We treat squared coefficients as a probability distribution
        p = np.array(c)**2 / (energy + 1e-20)
        feats[f'wav_entropy_{name}'] = -np.sum(p * np.log2(p + 1e-20))
        
        # 3. Wavelet Standard Deviation
        # (Measure of variation within this band)
        feats[f'wav_std_{name}'] = np.std(c)

    # --- FOD SPECIFIC WAVELET RATIOS ---
    sum_detail_energy = sum(feats[f'wav_energy_detail_{k}'] for k in range(1, decomp_level+1))
    approx_energy = feats['wav_energy_approx']
    
    feats['wav_detail_ratio'] = sum_detail_energy / (approx_energy + 1e-20)

    # Feature: High-Frequency Noise Index (Level 1 + Level 2)
    # These correspond to the highest frequencies (likely switching noise or arcing)
    high_freq_energy = feats['wav_energy_detail_1'] + feats['wav_energy_detail_2']
    feats['wav_high_freq_ratio'] = high_freq_energy / (total_wav_energy + 1e-20)
    
    return feats

# Building dataset

In [ ]:
records = []
files = glob.glob(f"{DATA_DIR}/C6--*.csv")

for fname in tqdm(files):
    labels = parse_filename(fname)

    try:

        df = pd.read_csv(fname, skiprows=4, usecols=['Time', 'Ampl'])

        if df.empty or 'Time' not in df or 'Ampl' not in df:
            print(f"Skipping empty or invalid file: {Path(fname).name}")
            continue

        t = df['Time'].to_numpy()
        x = df['Ampl'].to_numpy()
        feats = compute_features_from_array(t, x)
        rec = {**labels, **feats, 'file': Path(fname).name}
        records.append(rec)

    except pd.errors.EmptyDataError:
        print(f"Skipping empty file: {Path(fname).name}")
        continue
    except Exception as e:
        print(f"Error processing {Path(fname).name}: {e}")
        continue

feature_df_new = pd.DataFrame(records)

if os.path.exists(OUT_CSV):
    feature_df_old = pd.read_csv(OUT_CSV)
    feature_df = pd.concat([feature_df_old, feature_df_new], ignore_index=True)
    print(f"🔄 Appending {len(feature_df_new)} new rows to existing {OUT_CSV}")

else:
    feature_df = feature_df_new
    print(f"🆕 Creating new {OUT_CSV}")

feature_df.to_csv(OUT_CSV, index=False)
print(f"✅ Saved updated dataset to {OUT_CSV} (total rows: {len(feature_df)})")

In [ ]:
fname = 'dataset/can/C6--can--1--2--00000.csv'

df = pd.read_csv(fname, skiprows=4, usecols=['Time', 'Ampl'])

t = df['Time'].to_numpy()
x = df['Ampl'].to_numpy()

N = len(x)
fs = 1.0 / np.mean(np.diff(t))
print(f"Sampling Rate: {fs} Hz")

duration = N / fs
print(f"Segment Length: {N} samples ({duration} seconds)")
